# Review data cleaning (Amazon Reviews 2023 + Deceptive Opinion Spam)

Standard schema:

`review_id`, `user_id`, `product_id`, `rating`, `review_text`, `timestamp`, `label`, `source`

- **label**: `1` fake/deceptive, `0` genuine/truthful, missing → `NaN` (exported as empty in CSV)
- **source**: `"amazon"` or `"deceptive_spam"`
- **Amazon Reviews 2023** (Hugging Face `McAuley-Lab/Amazon-Reviews-2023`): fields include `user_id`, `asin`, `parent_asin`, `rating`, `title`, `text`, `timestamp` (Unix **milliseconds**).
- **Deceptive Opinion Spam** (Kaggle): use a combined CSV **or** point `SPAM_ROOT` at the extracted corpus folder (`.txt` files under paths containing `deceptive` / `truthful`).

In [14]:
import html
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

In [15]:
def clean_text(text):
    if pd.isna(text):
        return None

    text = str(text)
    text = html.unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


def safe_timestamp(series, default_date="2024-01-01 00:00:00"):
    ts = pd.to_datetime(series, errors="coerce")
    ts = ts.fillna(pd.Timestamp(default_date))
    return ts.astype(str)


def amazon_timestamp_to_str(series: pd.Series) -> pd.Series:
    """Amazon JSONL often stores Unix ms; read_json may coerce that to datetime64."""
    if pd.api.types.is_datetime64_any_dtype(series):
        ts = pd.to_datetime(series, errors="coerce")
        if ts.dt.tz is not None:
            ts = ts.dt.tz_convert(None)
    else:
        num = pd.to_numeric(series, errors="coerce")
        ts = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
        ok = num.notna()
        if ok.any():
            max_v = float(num.loc[ok].max())
            unit = "ms" if max_v >= 1e11 else "s"
            parsed = pd.to_datetime(num.loc[ok], unit=unit, errors="coerce", utc=True).dt.tz_convert(None)
            ts.loc[ok] = parsed
        bad = ts.isna()
        if bad.any():
            parsed = pd.to_datetime(series.loc[bad], errors="coerce", utc=True)
            if parsed.dt.tz is not None:
                parsed = parsed.dt.tz_convert(None)
            ts.loc[bad] = parsed
    ts = ts.fillna(pd.Timestamp("2024-01-01 00:00:00"))
    return ts.astype(str)


def build_review_text_from_amazon(row: pd.Series) -> Optional[str]:
    parts = []
    for col in ("title", "text", "reviewText", "review_text"):
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            parts.append(str(row[col]))
    if not parts:
        return None
    return clean_text(" ".join(parts))


def load_amazon(path: str | Path, nrows: int | None = None) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    suf = path.suffix.lower()
    if suf in {".jsonl", ".json"} or ".jsonl" in path.name:
        comp = "gzip" if path.suffix == ".gz" or str(path).endswith(".jsonl.gz") else None
        return pd.read_json(path, lines=True, compression=comp, nrows=nrows)

    if suf == ".csv":
        return pd.read_csv(path, nrows=nrows)

    raise ValueError(f"Unsupported Amazon input type: {path}")


def load_spam_csv(path: str | Path, nrows: int | None = None) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, nrows=nrows)


def load_spam_from_corpus_root(root: str | Path) -> pd.DataFrame:
    """Ott et al.-style layout: .txt reviews under folders with 'deceptive' or 'truthful' in the path."""
    root = Path(root)
    rows: list[dict] = []
    for fp in root.rglob("*.txt"):
        low = str(fp).lower()
        if "deceptive" in low:
            label = 1
        elif "truthful" in low:
            label = 0
        else:
            continue
        text = fp.read_text(encoding="utf-8", errors="replace").strip()
        rows.append({"text": text, "label": label, "source_path": str(fp)})
    return pd.DataFrame(rows)


def normalize_spam_labels(df: pd.DataFrame) -> pd.Series:
    if "label" in df.columns:
        s = df["label"]
        if s.dtype == object:
            smap = {
                "deceptive": 1,
                "truthful": 0,
                "fake": 1,
                "real": 0,
                "spam": 1,
                "ham": 0,
                "neg": 1,
                "pos": 0,
            }
            s = s.astype(str).str.strip().str.lower().map(smap).combine_first(pd.to_numeric(s, errors="coerce"))
        return pd.to_numeric(s, errors="coerce")

    for col in ("deceptive", "class", "is_fake", "is_deceptive"):
        if col not in df.columns:
            continue
        s = df[col]
        if s.dtype == object:
            smap = {"deceptive": 1, "truthful": 0, "true": 1, "false": 0, "yes": 1, "no": 0}
            s = s.astype(str).str.strip().str.lower().map(smap).combine_first(pd.to_numeric(s, errors="coerce"))
        return pd.to_numeric(s, errors="coerce")

    return pd.Series(np.nan, index=df.index, dtype="float64")

In [16]:
# --- Paths: adjust to your machine ---
DATA_DIR = Path("data")

# Amazon Reviews 2023 category file (JSONL gzip)
AMAZON_PATH = DATA_DIR / "Electronics.jsonl.gz"

# Deceptive opinion spam (CSV: deceptive, hotel, polarity, source, text)
SPAM_CSV_PATH = DATA_DIR / "deceptive-opinion.csv"
SPAM_ROOT = DATA_DIR / "deceptive-opinion-spam-corpus"  # set to None to skip folder ingest

# Safety for huge files while iterating in Jupyter
AMAZON_NROWS = 50_000  # None = all rows
SPAM_NROWS = None

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("AMAZON_PATH:", AMAZON_PATH)
print("SPAM_CSV_PATH:", SPAM_CSV_PATH)
print("SPAM_ROOT:", SPAM_ROOT)

AMAZON_PATH: data/Electronics.jsonl.gz
SPAM_CSV_PATH: data/deceptive-opinion.csv
SPAM_ROOT: data/deceptive-opinion-spam-corpus


## Optional: sample Amazon Reviews 2023 via Hugging Face `datasets`

Uncomment and run **once** you have `pip install datasets` and network access. This writes a small JSONL you can clean like any other file.

In [17]:
# from datasets import load_dataset

# subset = load_dataset(
#     "McAuley-Lab/Amazon-Reviews-2023",
#     "raw_review_All_Beauty",
#     split="full[:10000]",
#     trust_remote_code=True,
# )
# subset.to_json(DATA_DIR / "amazon_hf_sample.jsonl", orient="records", lines=True)
# print("Wrote", DATA_DIR / "amazon_hf_sample.jsonl")

In [18]:
amazon_df = load_amazon(AMAZON_PATH, nrows=AMAZON_NROWS)
amazon_df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True
1,1,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True
2,5,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True
3,5,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,2010-11-20 18:41:35.000,18,True
4,5,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,2023-02-17 02:39:41.238,0,True


In [19]:
print(amazon_df.columns.tolist())
print(amazon_df.shape)

['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
(50000, 10)


In [20]:
amazon_std = pd.DataFrame()

amazon_std["review_id"] = amazon_df.index.astype(str).map(lambda i: f"amz_{i}")

if "user_id" in amazon_df.columns:
    amazon_std["user_id"] = amazon_df["user_id"].astype(str)
elif "reviewerID" in amazon_df.columns:
    amazon_std["user_id"] = amazon_df["reviewerID"].astype(str)
else:
    amazon_std["user_id"] = "unknown_user"

if "parent_asin" in amazon_df.columns:
    amazon_std["product_id"] = amazon_df["parent_asin"].astype(str)
elif "asin" in amazon_df.columns:
    amazon_std["product_id"] = amazon_df["asin"].astype(str)
else:
    amazon_std["product_id"] = "unknown_product"

if "rating" in amazon_df.columns:
    amazon_std["rating"] = pd.to_numeric(amazon_df["rating"], errors="coerce")
elif "overall" in amazon_df.columns:
    amazon_std["rating"] = pd.to_numeric(amazon_df["overall"], errors="coerce")
else:
    amazon_std["rating"] = np.nan

amazon_std["review_text"] = amazon_df.apply(build_review_text_from_amazon, axis=1)

if "timestamp" in amazon_df.columns:
    amazon_std["timestamp"] = amazon_timestamp_to_str(amazon_df["timestamp"])
elif "unixReviewTime" in amazon_df.columns:
    ts = pd.to_datetime(pd.to_numeric(amazon_df["unixReviewTime"], errors="coerce"), unit="s", errors="coerce")
    amazon_std["timestamp"] = ts.fillna(pd.Timestamp("2024-01-01 00:00:00")).astype(str)
elif "reviewTime" in amazon_df.columns:
    amazon_std["timestamp"] = safe_timestamp(amazon_df["reviewTime"])
else:
    amazon_std["timestamp"] = "2024-01-01 00:00:00"

amazon_std["label"] = np.nan
amazon_std["source"] = "amazon"

amazon_std = amazon_std.dropna(subset=["review_text"])
amazon_std = amazon_std.drop_duplicates(subset=["review_text"])
amazon_std = amazon_std.reset_index(drop=True)

amazon_std.head()

,review_id,user_id,product_id,rating,review_text,timestamp,label,source
0,amz_0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B083NRGZMM,3,smells like gasoline going back first most off...,2022-07-18 22:58:37.948,NaN,amazon
1,amz_1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07N69T6TM,1,didn t work at all lenses loose broken these d...,2020-06-20 18:42:29.731,NaN,amazon
2,amz_2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B01G8JO5F2,5,excellent i love these they even come with a c...,2018-04-07 09:23:37.534,NaN,amazon
3,amz_3,AGGZ357AO26RQZVRLGU4D4N52DZQ,B001OC5JKY,5,great laptop backpack i was searching for a st...,2010-11-20 18:41:35.000,NaN,amazon
4,amz_4,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,B07CJYMRWM,5,best headphones in the fifties price range i v...,2023-02-17 02:39:41.238,NaN,amazon


In [21]:
if SPAM_CSV_PATH.exists():
    spam_df = load_spam_csv(SPAM_CSV_PATH, nrows=SPAM_NROWS)
elif SPAM_ROOT and SPAM_ROOT.exists():
    spam_df = load_spam_from_corpus_root(SPAM_ROOT)
else:
    raise FileNotFoundError(
        "Provide either data/deceptive-opinion.csv or extract the Kaggle corpus under data/deceptive-opinion-spam-corpus"
    )

spam_df.head()

,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...
1,truthful,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...
2,truthful,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...
3,truthful,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...
4,truthful,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...


In [22]:
print(spam_df.columns.tolist())
print(spam_df.shape)

['deceptive', 'hotel', 'polarity', 'source', 'text']
(1600, 5)


In [23]:
spam_std = pd.DataFrame()

spam_std["review_id"] = spam_df.index.astype(str).map(lambda i: f"spam_{i}")
spam_std["user_id"] = spam_df.index.astype(str).map(lambda i: f"user_spam_{i}")
if "hotel" in spam_df.columns:
    spam_std["product_id"] = spam_df["hotel"].astype(str).str.strip().str.lower()
else:
    spam_std["product_id"] = "hotel_review"
spam_std["rating"] = np.nan

text_col = None
for c in ("text", "review", "review_text", "Review", "TEXT"):
    if c in spam_df.columns:
        text_col = c
        break

if text_col is None:
    spam_std["review_text"] = None
else:
    spam_std["review_text"] = spam_df[text_col].apply(clean_text)

spam_std["label"] = normalize_spam_labels(spam_df)

if "timestamp" in spam_df.columns:
    spam_std["timestamp"] = safe_timestamp(spam_df["timestamp"])
else:
    spam_std["timestamp"] = "2024-01-01 00:00:00"

spam_std["source"] = "deceptive_spam"

spam_std = spam_std.dropna(subset=["review_text"])
spam_std = spam_std.drop_duplicates(subset=["review_text"])
spam_std = spam_std.reset_index(drop=True)

spam_std.head()

/var/folders/c_/xw25h9wx3pn7h872340s39q00000gq/T/ipykernel_94173/2179730833.py:116: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  s = s.astype(str).str.strip().str.lower().map(smap).combine_first(pd.to_numeric(s, errors="coerce"))


,review_id,user_id,product_id,rating,review_text,label,timestamp,source
0,spam_0,user_spam_0,conrad,NaN,we stayed for a one night getaway with family ...,0,2024-01-01 00:00:00,deceptive_spam
1,spam_1,user_spam_1,hyatt,NaN,triple a rate with upgrade to view room was le...,0,2024-01-01 00:00:00,deceptive_spam
2,spam_2,user_spam_2,hyatt,NaN,this comes a little late as i m finally catchi...,0,2024-01-01 00:00:00,deceptive_spam
3,spam_3,user_spam_3,omni,NaN,the omni chicago really delivers on all fronts...,0,2024-01-01 00:00:00,deceptive_spam
4,spam_4,user_spam_4,hyatt,NaN,i asked for a high floor away from the elevato...,0,2024-01-01 00:00:00,deceptive_spam


In [24]:
final_df = pd.concat([amazon_std, spam_std], ignore_index=True)

final_df = final_df[
    [
        "review_id",
        "user_id",
        "product_id",
        "rating",
        "review_text",
        "timestamp",
        "label",
        "source",
    ]
]

final_df.head()
print(final_df.shape)

(49627, 8)


In [25]:
print(final_df.isnull().sum())
print(final_df["source"].value_counts(dropna=False))
print(final_df["label"].value_counts(dropna=False))

review_id          0
user_id            0
product_id         0
rating          1596
review_text        0
timestamp          0
label          48031
source             0
dtype: int64
source
amazon            48031
deceptive_spam     1596
Name: count, dtype: int64
label
NaN    48031
1.0      800
0.0      796
Name: count, dtype: int64


In [26]:
amazon_std.to_csv("amazon_clean.csv", index=False)
spam_std.to_csv("spam_clean.csv", index=False)

final_df.to_csv("clean_reviews.csv", index=False)
final_df.to_json("clean_reviews.json", orient="records", lines=True)

print("Saved amazon_clean.csv, spam_clean.csv, clean_reviews.csv, clean_reviews.json")

Saved amazon_clean.csv, spam_clean.csv, clean_reviews.csv, clean_reviews.json
